In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from scipy.sparse import hstack
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("FAKE JOB POSTING DETECTION - MODEL TRAINING")
print("="*60)

FAKE JOB POSTING DETECTION - MODEL TRAINING


In [3]:
# Load the dataset
df = pd.read_csv('C://Users//venkat Reddy//Documents//Fake_job_detection//data//fake_job_postings.csv')

print("\n📊 Dataset Information:")
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['fraudulent'].value_counts())
print(f"\nFraudulent percentage: {(df['fraudulent'].sum() / len(df) * 100):.2f}%")

# Display first few rows
df.head()


📊 Dataset Information:
Dataset shape: (17880, 18)

Class distribution:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Fraudulent percentage: 4.84%


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [4]:
print("\n🔄 Preprocessing data...")

# Fill missing values
text_columns = ['company_profile', 'description', 'requirements', 'benefits', 
                'title', 'location', 'department', 'salary_range', 'employment_type',
                'required_experience', 'required_education', 'industry', 'function']

for col in text_columns:
    df[col] = df[col].fillna('')

# Combine textual fields
df['combined_text'] = (
    df['title'] + ' ' + 
    df['location'] + ' ' + 
    df['department'] + ' ' + 
    df['company_profile'] + ' ' + 
    df['description'] + ' ' + 
    df['requirements'] + ' ' + 
    df['benefits'] + ' ' +
    df['employment_type'] + ' ' +
    df['required_experience'] + ' ' +
    df['required_education'] + ' ' +
    df['industry'] + ' ' +
    df['function']
)

# Create additional features
df['text_length'] = df['combined_text'].apply(len)
df['word_count'] = df['combined_text'].apply(lambda x: len(x.split()))
df['has_company_logo'] = df['has_company_logo'].fillna(0).astype(int)
df['has_questions'] = df['has_questions'].fillna(0).astype(int)
df['telecommuting'] = df['telecommuting'].fillna(0).astype(int)
df['has_salary'] = df['salary_range'].apply(lambda x: 0 if x == '' else 1)

print("✅ Preprocessing completed!")


🔄 Preprocessing data...
✅ Preprocessing completed!


In [5]:
# Prepare features and target
X_text = df['combined_text']
X_numeric = df[['text_length', 'word_count', 'has_company_logo', 
                'has_questions', 'telecommuting', 'has_salary']]
y = df['fraudulent']

# Split data
X_train_text, X_test_text, X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📈 Data Split:")
print(f"Training set size: {len(X_train_text)}")
print(f"Test set size: {len(X_test_text)}")


📈 Data Split:
Training set size: 14304
Test set size: 3576


In [6]:
print("\n🔤 Applying TF-IDF vectorization...")
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Combine features
X_train_combined = hstack([X_train_tfidf, X_train_num.values])
X_test_combined = hstack([X_test_tfidf, X_test_num.values])

print(f"✅ Feature matrix shape: {X_train_combined.shape}")


🔤 Applying TF-IDF vectorization...
✅ Feature matrix shape: (14304, 5006)


In [9]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, learning_rate=0.1)
}

results = {}

print("\n" + "="*60)
print("TRAINING AND EVALUATING MODELS")
print("="*60)

for name, model in models.items():
    print(f"\n Training {name}...")
    model.fit(X_train_combined, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_combined)
    y_pred_proba = model.predict_proba(X_test_combined)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'predictions': y_pred
    }
    
    print(f"\n📊 {name} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC-AUC Score: {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(f"True Negatives: {cm[0][0]}, False Positives: {cm[0][1]}")
    print(f"False Negatives: {cm[1][0]}, True Positives: {cm[1][1]}")


TRAINING AND EVALUATING MODELS

 Training Logistic Regression...

📊 Logistic Regression Results:
Accuracy: 0.9533
ROC-AUC Score: 0.9882

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.96      0.97      3403
  Fraudulent       0.51      0.92      0.66       173

    accuracy                           0.95      3576
   macro avg       0.75      0.94      0.82      3576
weighted avg       0.97      0.95      0.96      3576


Confusion Matrix:
[[3250  153]
 [  14  159]]
True Negatives: 3250, False Positives: 153
False Negatives: 14, True Positives: 159

 Training Random Forest...

📊 Random Forest Results:
Accuracy: 0.9793
ROC-AUC Score: 0.9928

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99      3403
  Fraudulent       1.00      0.57      0.73       173

    accuracy                           0.98      3576
   macro avg       0.99      0.79      0.86    

In [10]:
import os

# Create model directory if it doesn't exist
model_dir = 'model'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f" Created '{model_dir}' directory")

# Select the best model
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']

print("\n" + "="*60)
print(f" BEST MODEL: {best_model_name}")
print("="*60)
print(f"Accuracy: {results[best_model_name]['accuracy']:.4f}")
print(f"ROC-AUC Score: {results[best_model_name]['roc_auc']:.4f}")

# Save the best model in the model folder
print("\n Saving model and vectorizer...")

with open(os.path.join(model_dir, 'best_model.pkl'), 'wb') as f:
    pickle.dump(best_model, f)

with open(os.path.join(model_dir, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)

# Save feature info
feature_info = {
    'numeric_features': ['text_length', 'word_count', 'has_company_logo', 
                         'has_questions', 'telecommuting', 'has_salary'],
    'model_name': best_model_name
}

with open(os.path.join(model_dir, 'model_info.pkl'), 'wb') as f:
    pickle.dump(feature_info, f)

print("\n Model training completed successfully!")
print(f"\n Files saved in '{model_dir}' folder:")
print(f"   - {model_dir}/best_model.pkl")
print(f"   - {model_dir}/tfidf_vectorizer.pkl")
print(f"   - {model_dir}/model_info.pkl")
print("\n Ready to run the Streamlit app!")


 BEST MODEL: Random Forest
Accuracy: 0.9793
ROC-AUC Score: 0.9928

 Saving model and vectorizer...

 Model training completed successfully!

 Files saved in 'model' folder:
   - model/best_model.pkl
   - model/tfidf_vectorizer.pkl
   - model/model_info.pkl

 Ready to run the Streamlit app!
